In [ ]:
%pip install -q python-dotenv
%pip install -q huggingface_hub
%pip install -q transformers

In [ ]:
from transformers.utils import logging

logging.set_verbosity_error()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = str(os.getenv("HF_TOKEN"))
if hf_token is None:
    raise ValueError('HF_TOKEN이 환경 변수에 설정되어 있지 않습니다.')
else:
    print(hf_token[:10])

In [ ]:
from huggingface_hub import login, whoami

login()
whoami()['name']

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation") # 최초 모델 다운로드시 시간이 걸림
generated = generator("I believe I can") # 약 1분 32초 걸림
print(generated)
print(generated[0]['generated_text'].replace("\n\n", " "))

In [ ]:
generator.model.name_or_path

In [ ]:
generated = generator(
    "I believe I can",
    max_new_tokens=10,
    num_return_sequences=3
)

for i, output in enumerate(generated, start=1):
    print(f"{i}: {output['generated_text'].replace("\n\n", " ")}")

In [ ]:
from transformers import pipeline

unmasker = pipeline(
    task="fill-mask",
    model="distilbert/distilroberta-base",
)

text = "I am going to <mask> to study."
entities = unmasker(text)

print(entities)
for entity in entities:
    print(f"{entity['token_str']}, {entity['score']:.4f}")

In [ ]:
unmasker = pipeline("fill-mask", "bert-base-uncased")
text = "I am going to [MASK] to study."
entities = unmasker(text)

for entity in entities:
    print(f"{entity['token_str']}, {entity['score']:.4f}")

**Exercise 1.1**

fill-mask 파이프라인을 사용하여 다음 문장의 빈칸을 예측하시오.

>The capital of Korea is _____.

상위 5개의 예측 결과를 출력하세요.

아래 두 모델의 출력 결과를 서로 비교해보세요.

* bert-base-uncased
* roberta-base

In [ ]:
from transformers import pipeline

def get_entities(model, text):
    unmasker = pipeline("fill-mask", model)
    entities = unmasker(text)
    return entities

entities = get_entities("bert-base-uncased", "The capital of Korea is [MASK].")
print("BERT:")
for entity in entities:
    print(f"{entity['token_str']}, {entity['score']:.4f}")
    
entities = get_entities("roberta-base", "The capital of Korea is <mask>.")
print("RoBERTa:")
for entity in entities:
    print(f"{entity['token_str']}, {entity['score']:.4f}")

In [ ]:
from transformers import pipeline

ner = pipeline(
    task="token-classification",
    aggregation_strategy="simple"
)
ner.model.name_or_path

In [ ]:
text = "My name is Joonion. I am teaching at KNU in Daegu, Korea."
entities = ner(text)
for entity in entities:
    print(f"{entity['word']}: {entity['entity_group']}, {entity['score']:.4f}")

In [ ]:
text = "안녕하세요? 저는 경북대학교 배준현 교수입니다. 제 전공은 컴퓨터 과학입니다."
entities = ner(text)
print(entities)

In [ ]:
from transformers import pipeline

text = "안녕하세요? 저는 경북대학교 배준현 교수입니다. 제 전공은 컴퓨터 과학입니다."

ner = pipeline(
    task="token-classification",
    model="monologg/koelectra-small-finetuned-naver-ner",
    aggregation_strategy="simple"
)
entities = ner(text)
for entity in entities:
    print(f"{entity['word']}: {entity['entity_group']}, {entity['score']:.4f}")

**Exercise 1.2**

아래에 주어진 문장으로 개체명 인식(NER)을 수행하세요.

>경북대학교 배준현 교수는 경산에 위치한 경일대학교에서 KDT14기 교육생들을 대상으로 전이 학습 기반 모델, 웹 기반 AI 서비스, 클라우드 기반 AI 서비스를 강의하였다. 이 강의는 Hugging Face와 LangChain을 활용한 인공지능 웹 서비스 개발을 목표로 진행되었고, 이 과정에서는 다양한 오픈 LLM 모델을 파인 튜닝하고, RAG를 기반으로 생성형 AI 서비스를 어떻게 개발하는지에 대한 실습 위주의 강의가 되었다.

다음 두 모델의 출력 결과를 서로 비교해보세요.

* monologg/koelectra-small-finetuned-naver-ner
* soddokayo/klue-roberta-large-klue-ner

In [ ]:
text = """
2026년 7월, 경북대학교 배준현 교수는 경산에 위치한 경일대학교에서 KDT14기 교육생들을 대상으로 전이학습 기반 모델, 웹 기반 AI 서비스, 클라우드 기반 AI 서비스를 강의하였다. 이 강의는 Hugging Face와 LangChain을 활용한 인공지능 웹 서비스 개발을 목표로 진행되었고, 이 과정에서는 다양한 오픈 LLM 모델을 파인 튜닝하고, RAG를 기반으로 생성형 AI 서비스를 어떻게 개발하는지에 대한 실습 위주의 강의가 되었다.
"""

from transformers import pipeline

ner = pipeline(
    task="token-classification",
    # model="monologg/koelectra-small-finetuned-naver-ner",
    model="soddokayo/klue-roberta-large-klue-ner",
    aggregation_strategy="simple"
)

entities = ner(text)
for entity in entities:
    print(f"{entity['word']}: {entity['entity_group']}, {entity['score']:.4f}")

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification")
classifier.model.name_or_path

In [ ]:
texts = [
    "I love this movie!",
    "I dislike this movie!"
]
results = classifier(texts)
print(results)
for text, result in zip(texts, results):
    print(f"{result['label']}: {result['score']}, {text}")

In [ ]:
texts = [
    "나는 이 음악이 너무 좋아.",
    "나는 이 음악이 너무 싫어.",
]
classifier = pipeline(
    "text-classification",
    model="searle-j/kote_for_easygoing_people",
)
results = classifier(texts)
for i, result in enumerate(results):
    print(f"{result['label']}: {result['score']}, {texts[i]}")

**Exercise 3.3**

아래 다섯 개의 문장으로 감성 분석을 수행하시오.

* 이 수업을 들은 건 내 인생 최고의 선택이야.
* 이 수업은 꽤 흥미로운 주제를 다뤘어.
* 수업 내용은 좋지만 설명이 조금 어렵다.
* 오늘 수업은 재미있긴 했지만 유익하진 않았어.
* 내가 들은 수업 중 이 수업이 진짜 최악이었어.

사전학습 모델로 다음 모델을 적용해 보시오.

* nlptown/bert-base-multilingual-uncased-sentiment

In [ ]:
texts = [
    "이 수업을 들은 건 내 인생 최고의 선택이야.",
    "이 수업은 상당히 흥미로운 주제를 다뤘어.",
    "수업 내용은 좋지만 설명이 조금 어렵다.",
    "오늘 수업은 재미있긴 했지만 유익하진 않았어.",
    "내가 들은 수업 중 이 수업이 진짜 최악이었어.",
]
classifier = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = classifier(texts)
for i, result in enumerate(results):
    print(f"{result['label']}: {result['score']}, {texts[i]}")

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier.model.name_or_path

In [ ]:
text = "This course teaches students how to write a novel."
labels = [
    "education", "technology", "healthcare", "sports", "politics"
]

results = classifier(text, labels)
print(results)
for label, score in zip(results['labels'], results['scores']):
    print(f"{label}: {score:.4f}")